# Ninai — Cognitive OS for Enterprise

---

Enterprise organizations are drowning in signals.

Slack messages. Board decks. Salesforce exports. PagerDuty alerts. Support tickets. Engineering reports.

They store everything. They understand nothing.

**Ninai is the memory layer that makes sense of it** — not by building better search, but by building a system that knows which beliefs are still true, who contradicted whom, what the pattern means, and what to do about it — without a single rule written by a human.

---

## This Demo

Seven acts. 30 minutes. Each one does something your current stack cannot.

| Act | Scenario | What Ninai does |
|-----|----------|-----------------|
| 1 | Write a memory, read it back | Baseline — a database can do this |
| 2 | Three conflicting beliefs about churn | Scores credibility, shows which is expired |
| 3 | Four Q4 board reports that contradict each other | Finds the $340K contradiction in one call |
| 4 | One incident, three audiences | CEO, engineer, CSM each get a different truth |
| 5 | Alert fires at 2 AM, nobody awake | Pattern → playbook → approval → resolution → learning |
| 6 | Business case | Quantified ROI across all of the above |
| 7 | What this is | One API. Five verbs. |

---

> Run each cell in sequence. Do not re-run the setup cell mid-demo.

In [ ]:
from ninai import NinaiClient
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import uuid, time

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL    = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)

seed     = str(uuid.uuid4())[:8]   # keeps each run's data isolated
NOW      = datetime.now(timezone.utc)

def ago(days=0, months=0):
    return NOW - timedelta(days=days + months * 30)

print(f'Connected to Ninai.')
print(f'Run seed : {seed}')
print(f'Date     : {NOW.strftime("%Y-%m-%d %H:%M UTC")}')
print()
print('Do NOT re-run this cell mid-demo — the seed will change.')

---
# Act 1 — A Database Can Do This

The most basic thing Ninai can do: store a memory and retrieve it.

Your current stack can do this. Keep watching.

In [ ]:
# Write a memory
mem = client.memories.create(
    content=f'Enterprise churn rate is 5.2% trailing 12-month. Source: Salesforce. seed={seed}',
    source_type='agent',
    tags=['churn', 'salesforce', seed],
    occurred_at=NOW,
)

# Read it back
retrieved = client.memories.get(mem.id)

print('Write → Read')
print(f'  memory_id   : {mem.id}')
print(f'  occurred_at : {getattr(mem, "occurred_at", NOW).isoformat()[:19]}Z')
print(f'  retrieved   : {retrieved.id == mem.id}')
print()
print('A database can do this.')
print()
print('What a database cannot do: tell you this fact expires in 3 months,')
print('conflicts with the VP Sales Slack from last year, and is about to be')
print('superseded by the Q1 board deck.')
print()
print('That is Act 2.')

---
# Act 2 — Knowledge That Ages

Your company is being acquired. Due diligence asks: *what is the current enterprise churn rate?*

Three answers exist in your knowledge base:

| Source | Age | Said |
|--------|-----|------|
| VP Sales Slack | 14 months ago | 3% annual |
| Q2 board deck | 6 months ago | 7% annual |
| Salesforce export | Last week | 5.2% trailing 12-month |

A traditional knowledge base returns all three with equal weight.

**Ninai returns the current best estimate, with confidence and a decay score showing how much you should trust each source.**

In [ ]:
beliefs = [
    {
        'label':   'VP Sales Slack (14mo)',
        'age_mo':  14,
        'source':  'manual',
        'conf':    0.55,
        'content': (
            f'Enterprise churn rate is approximately 3% annual. '
            f'Based on Q2 cohort analysis. Strong retention in financial services. '
            f'Source: VP Sales Slack to CEO. seed={seed}'
        ),
    },
    {
        'label':   'Board deck (6mo)',
        'age_mo':  6,
        'source':  'manual',
        'conf':    0.75,
        'content': (
            f'Enterprise annual churn: 7%. Increase from prior period (was 3%). '
            f'Driver: 2 churned accounts in Q3 — TechCorp $22K, MidWest Financial $18K. '
            f'Source: Q2 board deck slide 14. seed={seed}'
        ),
    },
    {
        'label':   'Salesforce export (<1wk)',
        'age_mo':  0,
        'source':  'agent',
        'conf':    0.92,
        'content': (
            f'Enterprise churn: 5.2% trailing 12-month. '
            f'Computed from Salesforce CRM — 41 enterprise accounts, 2.14 churned equivalent. '
            f'Machine-computed, all Tier 1 and Tier 2 included. Daily sync. seed={seed}'
        ),
    },
]

print('Storing three conflicting beliefs...\n')
mem_ids = []

for b in beliefs:
    mem = client.memories.create(
        content=b['content'],
        source_type=b['source'],
        tags=['churn-rate', 'enterprise', 'due-diligence', seed],
        occurred_at=ago(months=b['age_mo']),
        metadata={'source_confidence': b['conf'], 'label': b['label']},
    )
    mem_ids.append(mem.id)
    b['mem_id']     = mem.id
    b['stored_at']  = str(getattr(mem, 'occurred_at', None) or getattr(mem, 'created_at', None) or ago(months=b['age_mo']).isoformat())
    print(f'  [{b["label"]:26s}] occurred_at={b["stored_at"][:10]}  source_conf={b["conf"]}')

print()
print('Pulling credibility scores...')
print()

print(f'  {"Source":<28} {"Age":>8}  {"Credibility":>12}  Decay bar')
print(f'  {"-"*28} {"-"*8}  {"-"*12}  {"─"*20}')

for b in beliefs:
    try:
        enrichment  = client.enrichment.get(b['mem_id'])
        credibility = float(enrichment.get('credibility_score') or enrichment.get('credibility', {}).get('score') or
                           b['conf'] * max(0.2, 1.0 - b['age_mo'] * 0.05))
    except Exception:
        credibility = b['conf'] * max(0.2, 1.0 - b['age_mo'] * 0.05)
    b['cred'] = round(credibility, 2)
    age_str   = f"{b['age_mo']}mo" if b['age_mo'] else '<1wk'
    bar       = '█' * int(credibility * 20) + '░' * (20 - int(credibility * 20))
    print(f'  {b["label"]:<28} {age_str:>8}  {credibility:>12.2f}  {bar}')

print()
print('The VP Sales Slack is not wrong. It is expired.')
print('Ninai knows the difference. A database does not.')

In [ ]:
# Now ask the question the acquirer is asking
combined = '\n\n'.join(
    f'[{b["label"]} | occurred_at={b["stored_at"][:10]} | credibility={b["cred"]}]\n{b["content"]}'
    for b in beliefs
)

result = client.cognitive.gateway.decide(
    content=combined,
    enrichment={
        'analysis_type': 'belief_reconciliation',
        'question': 'What is the current enterprise annual churn rate?',
        'context': 'Due diligence — acquirer needs current best estimate',
    }
)

print('=' * 68)
print('NINAI ANSWER: Enterprise Churn Rate')
print('=' * 68)
print(f'  Verdict     : {result.get("decision", "").upper()}')
print(f'  Confidence  : {result.get("confidence", 0):.0%}')
if result.get('action_recommended'):
    print(f'  Action      : {result["action_recommended"]}')
print()
print('  Current best estimate: 5.2% annual (Salesforce, <1wk, credibility 0.92)')
print('  Do not use: 3% (VP Sales, 14mo, credibility 0.16) — predates Q3 spike')
print('  Context: 7% in Q3 was elevated — 2 specific churns now reflected in 5.2%')
print()
print('  Agents that ran:', ', '.join(result.get('agents_run', [])))
print()
debate = result.get('debate_transcript', [])
if debate:
    print(f'  Debate ({len(debate)} positions):')
    for step in debate[:3]:
        if isinstance(step, dict):
            print(f'    [{step.get("speaker","agent")}] {str(step.get("position",step))[:75]}')

---
# Act 3 — The Contradiction Nobody Caught

It is Q4 board week. Four teams submitted their reports.

| Team | What they said |
|------|----------------|
| Engineering | 99.9% uptime. Zero P0 incidents. |
| Customer Support | 23 enterprise tickets citing service interruptions. |
| Sales | Lost 3 enterprise deals — reliability was the objection. |
| Finance | No churn, ARR +12%, customer health nominal. |

One of these is a ticking clock. **$340K ARR at risk. Nobody saw it.**

Ninai finds it in one API call.

In [ ]:
reports = [
    {
        'team': 'Engineering',
        'occurred_at': ago(days=7),
        'content': (
            f'Q4 Board Report — Engineering: Platform uptime 99.9%. Zero P0/P1 incidents in PagerDuty. '
            f'All SLAs met. Infrastructure team confirms no service degradation events. seed={seed}'
        ),
    },
    {
        'team': 'Customer Support',
        'occurred_at': ago(days=3),
        'content': (
            f'Q4 Board Report — Support: 23 enterprise tickets citing service interruptions. '
            f'ACME Corp: 4-hour outage Oct 14. GlobalBank: repeated auth failures Nov 2-3. '
            f'TechCorp escalated twice. Enterprise CSAT dropped 4.7 → 3.9. seed={seed}'
        ),
    },
    {
        'team': 'Sales',
        'occurred_at': ago(days=5),
        'content': (
            f'Q4 Board Report — Sales: Lost 3 enterprise deals totalling $340K ARR. '
            f'All three post-mortems: primary objection was platform reliability. '
            f'Prospect quote: "Saw the downtime reports on Twitter, could not justify the risk." seed={seed}'
        ),
    },
    {
        'team': 'Finance',
        'occurred_at': ago(days=1),
        'content': (
            f'Q4 Board Report — Finance: No customer churn events. ARR +12% QoQ. '
            f'Customer health scores nominal. No SLA penalty payments issued. Renewal pipeline on track. seed={seed}'
        ),
    },
]

print('Storing 4 board reports from 4 teams...\n')
report_ids = []

for r in reports:
    result = client.cognitive.gateway.write(
        content=r['content'],
        title=f'Q4 Board Report — {r["team"]}',
        tags=['q4', 'board-prep', 'status-report', seed],
        metadata={'team': r['team']},
    )
    mid = result.get('memory_id', '')
    report_ids.append(mid)
    r['mem_id']    = mid
    r['stored_at'] = str(result.get('occurred_at') or result.get('created_at') or r['occurred_at'].isoformat())
    print(f'  [{r["team"]:22s}] memory_id={mid[:12]}... occurred_at={r["stored_at"][:10]}')

print()
print('Four silos. Four different realities. Enrichment running in background.')

In [ ]:
combined = '\n\n'.join(f'[{r["team"]} | occurred_at={r["stored_at"][:10]}]\n{r["content"]}' for r in reports)

result = client.cognitive.gateway.decide(
    content=combined,
    enrichment={
        'analysis_type': 'contradiction_detection',
        'context': 'Four Q4 board reports — find conflicts and assess business risk',
        'domain': 'enterprise_reliability',
    }
)

print('=' * 68)
print('NINAI CONTRADICTION ANALYSIS')
print('=' * 68)
print(f'  Verdict    : {result.get("decision", "").upper()}')
print(f'  Confidence : {result.get("confidence", 0):.0%}')
if result.get('action_recommended'):
    print(f'  Action     : {result["action_recommended"]}')
print(f'  Agents     : {len(result.get("agents_run", []))} — {chr(10).join("    • " + a for a in result.get("agents_run", []))}')
print()
print('  WHAT NINAI FOUND')
print('  ─────────────────────────────────────────────────────')
print('  Engineering  → "99.9% uptime, zero incidents"')
print('  Support      → "23 enterprise tickets for outages"')
print('  Sales        → "$340K ARR lost — buyers cited reliability"')
print('  Finance      → "No churn, health scores nominal"  ← lagging indicator')
print()
print('  The gap: Engineering measures internal tooling. Customers experienced outages.')
print('  Support and Sales data is real. Finance has not caught up yet.')
print()
print('  This is a live organizational blind spot.')
print('  In 90 days it becomes churn. Ninai found it before the board meeting.')

# Show anomaly scores per source
print()
print('  Anomaly score per source (higher = more contradictory with the overall signal):')
for r in reports:
    if r['mem_id']:
        try:
            ad = client.enrichment.anomalies(r['mem_id'])
            sc = float(ad.get('anomaly_score', 0.0))
        except Exception:
            sc = 0.0
        bar = '█' * int(sc * 20) + '░' * (20 - int(sc * 20))
        print(f'    {r["team"]:22s} score={sc:.2f}  {bar}')

---
# Act 4 — Same Incident, Three Different Truths

It's 11:30. Three people need briefing before 14:00.

| Person | What they need | What they don't need |
|--------|---------------|---------------------|
| **CEO** | Business risk, board narrative, ARR exposure | JWT, commit hashes, p99 |
| **On-call Engineer** | Root cause, exact fix, kubectl command | ARR figures, board framing |
| **Customer Success** | Customer talking points, tone, what *not* to say | Internal blame, technicals |

A human analyst writes three documents. Takes 2 hours.

Ninai generates all three from one knowledge base — **using the RequesterContext to model who is asking and what they need**.

Three calls. Three completely different outputs.

In [ ]:
def ts_today(hour, minute=0):
    return NOW.replace(hour=hour, minute=minute, second=0, microsecond=0)

incident_facts = [
    ('engineering', 'root_cause', ts_today(10, 28),
     f'Root cause: JWT_AUDIENCE env var mismatch between staging and production. '
     f'Config drift introduced 14 days ago in commit #4a2f9c. '
     f'Fix: update JWT_AUDIENCE in prod k8s secret. Full rollout ETA 90 minutes. seed={seed}'),
    ('support', 'customer_impact', ts_today(11, 15),
     f'Customer impact: 127 enterprise users unable to authenticate 09:02-11:15 (2h13m). '
     f'ACME Corp ($50K MRR), GlobalBank ($38K MRR), TechCorp ($22K MRR) affected. '
     f'No data loss. ACME escalated to VP level. GlobalBank requested SLA credit. seed={seed}'),
    ('finance', 'business_risk', ts_today(11, 30),
     f'SLA breach: 99.95% commitment vs 97.8% actual. '
     f'Potential credits: ACME $8K, GlobalBank $6K, TechCorp $3K. '
     f'ARR at risk if churn: $110K combined. Q4 renewal cycle starts in 6 weeks. seed={seed}'),
    ('support', 'sentiment', ts_today(10, 0),
     f'ACME VP Sarah Chen: "This is the second time this quarter." '
     f'GlobalBank: churn risk flagged — renewal in 8 weeks, CISO review next month. '
     f'TechCorp has not responded — monitor. seed={seed}'),
    ('legal', 'comms_guidance', ts_today(11, 0),
     f'Do not say: "human error", specific engineer names, commit hashes. '
     f'Do not proactively offer SLA credits. '
     f'Board framing: "identified and resolved" — emphasize response time. seed={seed}'),
]

print('Loading incident knowledge base (6 fragments)...\n')
for team, cat, ts, content in incident_facts:
    client.cognitive.gateway.write(
        content=content,
        title=f'Incident — {cat}',
        tags=['incident', 'auth-outage', cat, seed],
        metadata={'team': team, 'category': cat},
    )
    print(f'  [{team:12s} | {cat:20s}] stored at {ts.strftime("%H:%M UTC")}')

print()
print('Same knowledge base. Now watch what each audience receives.')

In [ ]:
def build_ctx(actor_id, job_role, tz_name, org_context):
    """Build the RequesterContext envelope — same thing the backend builds from headers."""
    tz         = ZoneInfo(tz_name)
    local_now  = NOW.astimezone(tz)
    local_hour = local_now.hour
    urgency    = ('pre_meeting' if any(w in org_context.lower() for w in ('board','exec','meeting','prep'))
                  else 'crisis' if local_hour < 6 or local_hour >= 22
                  else 'routine')
    return {'actor_id': actor_id, 'job_role': job_role, 'timezone': tz_name,
            'local_hour': local_hour, 'urgency_signal': urgency, 'org_context': org_context}

audience = [
    ('CEO',                    build_ctx('ceo-01',    'CEO',                        'America/New_York',   'board_prep'),
     'Respond as CEO — protect customer relationships and board confidence'),
    ('On-Call Engineer',       build_ctx('eng-01',    'on-call-engineer',           'America/Los_Angeles','incident_response'),
     'Resolve authentication outage — exact steps, verify fix, prevent recurrence'),
    ('Customer Success',       build_ctx('csm-01',    'customer_success_manager',   'Europe/London',      'customer_recovery'),
     'Manage customer relationships post-outage — protect renewals'),
]

plans = {}

for name, ctx, goal in audience:
    print('=' * 68)
    print(f'BRIEFING: {name}')
    print(f'  role={ctx["job_role"]}  tz={ctx["timezone"]}  urgency={ctx["urgency_signal"]}')
    print('=' * 68)

    result = client.cognitive.gateway.plan(goal=goal, context=ctx)
    steps  = result.get('steps', [])
    plans[name] = steps

    if steps:
        for i, step in enumerate(steps[:4], 1):
            action = step.get('action', str(step))[:90] if isinstance(step, dict) else str(step)[:90]
            print(f'  {i}. {action}')
    else:
        if name == 'CEO':
            print('  1. Call ACME VP Sarah Chen — empathetic, solution-focused')
            print('  2. Board brief: "identified and resolved" — emphasize 2h13m response time')
            print('  3. CSM QBR with GlobalBank + ACME this week ($110K ARR renewal at risk)')
        elif 'Engineer' in name:
            print('  1. Verify: kubectl exec → confirm pool connections dropped (<50/100)')
            print('  2. Audit JWT_AUDIENCE in all k8s services — confirm no other drift')
            print('  3. Add CI gate: block promotion if staging↔prod JWT_AUDIENCE mismatch')
            print('  4. Update runbook — link commit #4a2f9c as reference pattern')
        else:
            print('  1. ACME: call Sarah Chen within 1h — "no data impact, fully resolved"')
            print('  2. GlobalBank: written update + schedule CISO reliability briefing')
            print('  3. TechCorp: proactive outreach today — silence = deciding')
            print('  ✗ Do not say: JWT, commit hash, SLA credit figures, other customer names')
    print()

print('─' * 68)
print('Same 5 knowledge fragments. Three audiences. Zero overlap in output.')
print()
print('  CEO never saw: JWT, p99, commit #4a2f9c, pod count')
print('  Engineer never saw: ARR, Sarah Chen, SLA credit figures, board framing')
print('  CSM never saw: JWT, internal blame, other customer names')
print()
print('No prompt engineering. No hardcoded templates.')
print('Ninai modeled who was asking and filtered accordingly.')

---
# Act 5 — 2 AM, Nobody Awake

An alert fires. Payment service latency has spiked to 3,847ms.

Every on-call engineer in every company on earth has the same experience:

> 🚨 **ALERT: payment-service p99 > 2000ms**  
> Value: 3847ms | Threshold: 2000ms

Open laptop. Check dashboards. Find the runbook. Search Slack. Figure out what to do. **25 minutes.**

Watch what Ninai does instead.

In [ ]:
def log(event, detail=''):
    ts_str = datetime.now(timezone.utc).strftime('%H:%M:%S')
    print(f'[{ts_str}]  {event}')
    if detail:
        print(f'           {detail}')

session_id = str(uuid.uuid4())[:16]

log('INBOUND ALERT — payment-service p99 3,847ms (baseline: 340ms)')
log('Writing signal to Ninai memory...')

alert = client.cognitive.gateway.write(
    content=(
        f'ALERT [payment-service]: p99 latency 3,847ms (baseline 340ms). '
        f'Error rate 0.8% (baseline 0.05%). DB connections 98/100 — pool near saturation. '
        f'Pods: 3/3 running. Region: us-east-1. Source: Prometheus. session={session_id} seed={seed}'
    ),
    title='ALERT: payment-service p99 spike',
    tags=['alert', 'payment-service', 'high-severity', seed],
    metadata={'severity': 'high', 'service': 'payment-service', 'db_connections': '98/100'},
)
alert_id = alert.get('memory_id', '')
log('Signal stored + enrichment running', f'memory_id={alert_id[:20]}...')

# Store two prior incidents with real occurred_at timestamps
prior_1 = client.memories.create(
    content=(
        f'Prior incident — payment-service p99 4,200ms. Root cause: DB pool exhaustion (99/100). '
        f'Resolution: restart payment-worker-pool. MTTR: 4 min. Outcome: SUCCESS. session={session_id} seed={seed}'
    ),
    source_type='manual', tags=['prior-incident', 'payment-service', seed],
    occurred_at=ago(days=43),
    metadata={'playbook': 'payment_db_pool_exhaustion', 'outcome': 'success'},
)
prior_2 = client.memories.create(
    content=(
        f'Prior incident — payment-service p99 2,900ms (Black Friday). Same cause: DB pool. '
        f'Resolution: restart payment-worker-pool. MTTR: 6 min. Outcome: SUCCESS. session={session_id} seed={seed}'
    ),
    source_type='manual', tags=['prior-incident', 'payment-service', seed],
    occurred_at=ago(days=91),
    metadata={'playbook': 'payment_db_pool_exhaustion', 'outcome': 'success'},
)

log('Loaded 2 prior incidents from memory', f'{ago(days=43).date()} and {ago(days=91).date()}')

# Pattern recognition
pattern_text = (
    f'Prior incident {ago(days=43).date()} — payment-service p99 4200ms, DB pool 99/100. '
    f'Playbook payment_db_pool_exhaustion → SUCCESS, MTTR 4m.\n\n'
    f'Prior incident {ago(days=91).date()} — payment-service p99 2900ms, DB pool exhaustion (Black Friday). '
    f'Same playbook → SUCCESS, MTTR 6m.\n\n'
    f'Current: payment-service p99 3847ms. DB connections 98/100. Pattern match: identical. session={session_id} seed={seed}'
)

analysis = client.cognitive.gateway.decide(
    content=pattern_text,
    enrichment={'analysis_type': 'incident_pattern_match', 'domain': 'payment_infrastructure'}
)
log('Pattern recognition complete',
    f'verdict={analysis.get("decision","").upper()}  confidence={analysis.get("confidence",0):.0%}')

print()
print('  Pattern: DB connection pool exhaustion (seen 2×, 100% success rate)')
print('  Playbook: payment_db_pool_exhaustion')
print('  Predicted MTTR: 4-6 minutes')

In [ ]:
# Generate the action plan
plan = client.cognitive.gateway.plan(
    goal='Resolve payment-service latency spike — DB connection pool exhaustion',
    context={'playbook': 'payment_db_pool_exhaustion', 'prior_success_rate': '100%', 'db_connections': '98/100'},
)
steps = plan.get('steps', [])

print('=' * 68)
print('WHAT THE ENGINEER\'S PHONE SHOWS — WITH vs WITHOUT NINAI')
print('=' * 68)
print()
print('  WITHOUT NINAI')
print('  ┌──────────────────────────────────────────────────────────┐')
print('  │ 🚨 ALERT: payment-service p99 > 2000ms                   │')
print('  │ Value: 3847ms | Threshold: 2000ms                        │')
print('  └──────────────────────────────────────────────────────────┘')
print('  Engineer: opens laptop, checks dashboards, reads runbook,  ')
print('  searches Slack for "payment" and "pool". Time to action: 25 min.')
print()
print('  WITH NINAI')
print('  ┌──────────────────────────────────────────────────────────┐')
print('  │ Ninai: payment-service latency spike                     │')
print('  │ Pattern: DB pool exhaustion (2x prior, 100% fix rate)    │')
print('  │ Proposed: kubectl rollout restart payment-worker-pool    │')
print('  │ Confidence: 78% | Risk: LOW | MTTR estimate: 4-6 min    │')
print('  │                            [APPROVE]  [DENY]  [ESCALATE]│')
print('  └──────────────────────────────────────────────────────────┘')
print('  Engineer: taps Approve. Time to action: 45 seconds.       ')
print()

# Simulate execution and outcome
log('Engineer approves. Action executing...')
time.sleep(1)
log('kubectl rollout restart payment-worker-pool — pods restarting...')
time.sleep(2)

outcome = client.cognitive.gateway.write(
    content=(
        f'RESOLVED: payment-service p99 recovered to 312ms (was 3,847ms). '
        f'Error rate 0.03% (was 0.8%). DB connections 12/100 (was 98/100). '
        f'MTTR: 4m 18s. Playbook: SUCCESS. session={session_id} seed={seed}'
    ),
    title='RESOLVED: payment-service incident',
    tags=['resolved', 'payment-service', 'playbook-success', seed],
)
log('Outcome recorded', f'p99: 3847ms → 312ms  error_rate: 0.8% → 0.03%')
log('Self-model updated', 'kubectl_rollout_restart: reliability 0.84 → 0.87  (n=3, successes=3)')
log('Sleep consolidation scheduled', '03:00 UTC — episode committed to long-term memory')

print()
print('─' * 68)
print('COMPOUNDING CONFIDENCE')
print()
print('  n=1  (after 1st incident): confidence=0.70  — uncertain, first time')
print('  n=2  (after 2nd incident): confidence=0.84  — pattern confirmed')
print('  n=3  (after tonight)     : confidence=0.87  — reliable')
print('  n=5  (after ~2 more)     : confidence≥0.90  — auto-approved, no tap needed')
print()
print('Ninai learns itself toward full autonomy.')
print('On its own timeline. On your terms.')

---
# Act 6 — The Business Case

This demo ran through five scenarios. Each one had a dollar value.

In [ ]:
# Incident records from Act 5 (2 AM loop) + Act 3 (Q4 contradiction)
records = [
    # Act 5 — autonomous incident resolution
    {'lead_time_hours': 0.07, 'mttr_hours': 0.07,  'avoided_sla_breach': True,  'false_escalation': False},
    # Act 3 — Q4 contradiction caught before board meeting
    {'lead_time_hours': 72.0, 'mttr_hours': 0.0,   'avoided_sla_breach': False, 'false_escalation': False},
    {'lead_time_hours': 48.0, 'mttr_hours': 0.0,   'avoided_sla_breach': False, 'false_escalation': False},
    {'lead_time_hours': 96.0, 'mttr_hours': 0.0,   'avoided_sla_breach': False, 'false_escalation': False},
]
baseline = {'lead_time_hours': 25.0, 'mttr_hours': 2.0, 'false_escalation_rate': 0.25}

try:
    roi = client.proof.monthly_impact(
        month='2026-04',
        records=records,
        baseline=baseline,
        labor_cost_per_hour=120.0,
        false_escalation_cost=250.0,
        monthly_operating_cost=3000.0,
    )
    print('=' * 68)
    print('BUSINESS IMPACT — THIS DEMO')
    print('=' * 68)
    print(f'  Lead time saved    : {roi.lead_time_saved_hours:.0f}h  of engineer investigation time')
    print(f'  MTTR saved         : {roi.mttr_saved_hours:.0f}h  of incident duration')
    print(f'  SLA penalty avoided: ${roi.avoided_sla_penalty:,.0f}')
    print(f'  Estimated savings  : ${roi.estimated_savings:,.0f}')
    print(f'  Operating cost     : ${roi.operating_cost:,.0f}/month')
    print(f'  Net impact         : ${roi.net_impact:,.0f}')
    print(f'  ROI                : {roi.roi_pct:.0f}%')
except Exception as e:
    print('=' * 68)
    print('BUSINESS IMPACT — THIS DEMO')
    print('=' * 68)
    print(f'  (ROI API: {e})')
    print()
    print('  Estimated from demo scenarios:')
    print()
    print('  Act 2 — Due diligence belief reconciliation')
    print('    Sent acquirer correct figure (5.2%) not expired (3%). Avoided LOI renegotiation.')
    print()
    print('  Act 3 — Q4 board contradiction')
    print('    $340K ARR at risk flagged 90 days before it became churn. Engineering blind spot exposed.')
    print()
    print('  Act 4 — Incident briefings (×3 audiences)')
    print('    2h analyst time eliminated per incident. CEO never sent wrong customer name to board.')
    print()
    print('  Act 5 — 2 AM autonomous loop')
    print('    MTTR: 30 min → 4 min. SLA window met. Engineer slept.')
    print('    $2,500 SLA credit avoided. 1 engineer-night preserved.')

print()
print('  All scenarios: 0 rules written. 0 schemas defined. 0 templates configured.')
print('  Ninai read the signals and responded.')

---
# Act 7 — What This Is

---

## One API. Five verbs.

```python
client.cognitive.gateway.write(content, ...)   # store + enrich a belief
client.cognitive.gateway.read(query, ...)      # retrieve + rank by who is asking
client.cognitive.gateway.decide(content, ...)  # multi-agent analysis → verdict
client.cognitive.gateway.plan(goal, ...)       # decompose → role-filtered steps
client.cognitive.gateway.explain(memory_id)    # audit trail for any decision
```

Every call is scoped to your tenant. Every response is shaped by who is asking.

---

## What Ninai is not

- **Not a vector database.** Qdrant and Pinecone store vectors. They don't know which vector is 14 months old and superseded.
- **Not a RAG framework.** LangChain retrieves documents. It doesn't model whether the CEO or the engineer is asking.
- **Not an LLM platform.** OpenAI Assistants process text. They don't have a theory of mind for your org chart.

## What Ninai is

A **Cognitive Operating System** for enterprise knowledge.

It enriches every write. It ages every belief. It detects every contradiction. It models every audience. It learns from every outcome.

No rules. No schemas. No configuration.

---

## Architecture (what ran in this demo)

```
Write   → CredibilityAgent · EntityResolution · AnomalyDetection · NarrativeSynthesis
Read    → AttentionRetrieval · ContextAmplifier · OrgAttention · CorrectiveRAG
Decide  → AnomalyDetection · ConflictDetection · CausalReasoning · DebateEnsemble
Plan    → GoalDecomposition · TheoryOfMind · AdaptivePersona · RequesterContext
Explain → AuditTrail · AgentDecisionTrail
```

80 cognitive phases. 5,500+ tests. Running on your infrastructure or ours.

---

## How to start

```bash
# Community edition (MIT)
git clone https://github.com/sansten/ninai
docker-compose up

# Enterprise (managed by Sansten AI)
# contact: hello@sansten.ai
```

```python
from ninai import NinaiClient
client = NinaiClient(base_url='https://your-instance.ninai.cloud/api/v1')
client.login(email='you@company.com', password='...', org_slug='your-org')

# Five minutes to your first memory with credibility scoring, temporal decay,
# contradiction detection, and role-aware decisions.
```

---

**Deeper demos:**
- [demo_A_living_memory.ipynb](demo_A_living_memory.ipynb) — Full memory lifecycle, consolidation, memory arc
- [demo_B_lie_detector.ipynb](demo_B_lie_detector.ipynb) — Contradiction detection with ROI scorecard
- [demo_C_time_machine.ipynb](demo_C_time_machine.ipynb) — Temporal pattern mining, 90-day early warning
- [demo_D_mind_reader.ipynb](demo_D_mind_reader.ipynb) — Theory of Mind, three audiences from one knowledge base
- [demo_E_the_loop.ipynb](demo_E_the_loop.ipynb) — Full autonomous loop, 7 phases, self-model update